# TURN-Mapper — stage S2 fine-tuning on **Kaggle** (docs/MASTER_PLAN.md, W3)

Starts from the CubiCasa5K checkpoint (S1, test mIoU 0.65) and fine-tunes on:
- **ResPlan** (17K residential plans as vector geometry, CC BY 4.0), drawn by us in 8 styles — thin-line, double-line, hatched, CAD, scanned, colour, grey, black walls — with exact masks;
- **procedural institutional floors** (double-loaded corridors, lecture halls, stair cores, lifts, washrooms, lobbies with voids);
- a 1,500-plan **CubiCasa replay** so the S1 skills are kept.

**Before Run All:**
1. Settings → Accelerator → **GPU T4 x2** (one is used); Settings → Internet → **On**.
2. Add Input → *Notebooks* → **turn-segmenter-cubicasa-pretrain** (your S1 run; provides `turn_seg_cubicasa.pt`).
3. Add Input → Datasets → **qmarva/cubicasa5k** (replay; optional) and **resplan/resplan** (optional — otherwise cloned from GitHub).

Takes about 4–5 h. Outputs go to `/kaggle/working/out_s2` → **Save Version** to keep them. Weights never go into git.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/ZakiNabeel/Turn-Mapper.git /kaggle/working/Turn-Mapper
%cd /kaggle/working/Turn-Mapper
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1 || echo "tesseract only needed for the pipeline check"
!pip -q install -e ".[ml]"
!git log --oneline -1

## Preview the training drawings (check before spending GPU hours)

In [ ]:
import sys, glob, cv2, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "/kaggle/working/Turn-Mapper/src")
from turn_mapper.learned.institutional import generate
from turn_mapper.learned.styles import render, STYLES
from turn_mapper.learned.annotate import colorize
rng = np.random.default_rng(0)
fig, ax = plt.subplots(2, 4, figsize=(22, 9))
for a, st in zip(ax.ravel(), STYLES):
    m, ex = generate(rng); a.imshow(render(m, ex, st, rng)); a.set_title(st); a.axis("off")
plt.show()

## Run S2 (data preparation, fine-tuning, evaluation)

In [ ]:
!bash scripts/kaggle_finetune_s2.sh 20 1 3000 2>&1 | tail -80

## Results

In [ ]:
!ls -la /kaggle/working/out_s2
!cat /kaggle/working/out_s2/eval_s2_*.json | head -80